# Imports

In [110]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
import json
import re
import tqdm
from utils import get_examples_from_df
from pathlib import Path
from datetime import datetime as dt

# EDA

In [111]:
base_path = Path().cwd()

In [133]:
# # read data and rename columns
# df = pd.read_csv("data/requirements_full.csv")
# df.columns = ["requirement", "s1", "s2", "s3", "s4", "s5", "s6"]
df = pd.read_excel("data/requirements.xlsx")

In [134]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Drop Sensor

In [135]:
# # drop vihecal speed sensor
# df = df.loc[df["s6"]!=1]
# # df.drop(columns="s6", inplace=True)

# Find Examples

In [137]:
N_EXAMPLES = 3

indexes_to_drop, examples = get_examples_from_df(df, N_EXAMPLES)

In [138]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e1 in examples.values():
    for e2 in e1:
        examples_txt += f"Requirement: {e2[0]}\n"
        examples_txt += f"Vector: {e2[1]}\n"
        examples_txt += "\n"

In [139]:
print("\n".join(examples_txt.split("\n")[:8]))

Requirement: Should the primary control system fail, the accelerator pedal must enter a fail-safe mode, allowing for limited but controlled throttle response
Vector: [1,0,0,0,0]

Requirement: The system must log data relevant to accelerator pedal operation to assist in troubleshooting and fault analysis
Vector: [1,0,0,0,0]

Requirement: Vehicle acceleration must be linear and proportional to the displacement of the accelerator pedal to provide a smooth and controlled driving experience
Vector: [1,0,0,0,0]


In [121]:
# drop indexes used in examples
df.drop(index=indexes_to_drop, inplace=True)

# LLM

In [122]:
from prompts.SystemPrompts import SystemPrompt
from prompts.Sensors import Sensors
from prompts.UserPrompt import UserPrompt

In [123]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

In [124]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.0
)

# Run for all Requirements

In [125]:
def parse_result(res):
    '''Parse the LLM result'''
    pattern = r"\[(.*?)\]"
    vec = re.findall(pattern, res)[0]
    return f"[{vec}]".replace(" ", "")


def run_all_reqs(instance):
    # system prompt
    messages = [
        {'role': 'system',
        'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt)}
    ]

    result = {}

    result["idx"] = instance[0]
    result["requirement"] = instance[1].iloc[0]
    result["true_vector"] = "[" + ",".join(map(str, instance[1].iloc[1:])) + "]"

    # add user prompt
    messages.append({"role":"user", "content":UserPrompt.format(req=result["requirement"])})

    # run LLM
    response = llm.invoke(messages)
    result["ai_response"] = response.content
    result["pred_vector"] = parse_result(result["ai_response"])

    result["accuracy"] = result["pred_vector"] == result["true_vector"]

    result["ai_token_usage"] = response.response_metadata["token_usage"]

    return result

In [126]:
results = [run_all_reqs(i) for i in tqdm.tqdm(df.iterrows())]

217it [03:13,  1.12it/s]


In [127]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["ai_token_usage"]["total_tokens"]
    total_completion_tokens += r["ai_token_usage"]["completion_tokens"]

number_of_reqs = len(results)
accuracy /= len(results)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [128]:
accuracy

0.8387096774193549

In [129]:
# save results
time = dt.now()

results_path = "results/conv_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=llm.deployment_name,
    examples=N_EXAMPLES,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "responses": results}, f, indent=4)